# Variance Reduction

Variance reduction techniques improve Monte Carlo simulations by reducing estimator variance without increasing the number of simulated paths.

This notebook demonstrates variance reduction for Asian options using the **control variates method**:

- Compute the price of a Asian option based on **geometric averages**
- Estimate the geometric Asian option price via **Monte Carlo simulation**
- Apply **control variates** to price an arithmetic Asian option
- Evaluate the efficiency and accuracy of the method under different parameter settings (number of paths, strike price, monitoring dates)

## 1. Analytical Solution for an Asian Call Option with Geometric Averages

Under the Black–Scholes model, the stock price follows the risk–neutral SDE

$$
dS(t) = rS(t)dt + \sigma S(t)dW(t)
$$

whose solution is

$$
S(T) = S(t_0)\exp\left((r-\tfrac{1}{2}\sigma^2)T + \sigma \sqrt{T}Z \right),
$$

where  

- $r$ = risk-free rate  
- $\sigma$ = volatility  
- $Z \sim N(0,1)$  

Hence $S(T)$ is **lognormally distributed**. Assume the stock price is observed $N$ times before maturity

$$
t_i = \frac{iT}{N}, \quad i=0,\dots,N
$$

The **geometric average price** is

$$
\tilde{A}_N =
\left(\prod_{i=0}^{N} S(t_i)\right)^{\frac{1}{N+1}}
$$

The payoff of a geometric Asian call is

$$
(\tilde{A}_N - K)^+
$$

Using the ratio representation of prices,

$$
\prod_{i=1}^{N} S(t_i)
=
\frac{S(t_N)}{S(t_{N-1})}
\left(\frac{S(t_{N-1})}{S(t_{N-2})}\right)^2
\cdots
\left(\frac{S(t_1)}{S(t_0)}\right)^N S(t_0)^{N}
$$

From the GBM solution,

$$
\frac{S(t_i)}{S(t_{i-1})}
=
\exp\left(
(r-\tfrac{1}{2}\sigma^2)\frac{T}{N}
+
\sigma\sqrt{\frac{T}{N}}X_i
\right)
$$

where

$$
X_i \sim N(0,1)
$$

Taking the logarithm of the normalized geometric average

$$
\log\left(
\frac{\tilde{A}_N}{S(t_0)}
\right)
=
\frac{1}{N+1}
\sum_{i=1}^{N}
i
\left[
(r-\tfrac{1}{2}\sigma^2)\frac{T}{N}
+
\sigma\sqrt{\frac{T}{N}}X_i
\right]
$$

and using

$$
1 + 2 + \cdots + N = \frac{N(N+1)}{2}
$$

we obtain

$$
\log\left(\frac{\tilde{A}_N}{S(t_0)}\right)
=
(r-\tfrac{1}{2}\sigma^2)\frac{T}{2}
+
\sigma\sqrt{\frac{T}{N}}
\frac{\sum_{i=1}^N iX_i}{N+1}
$$

Since $X_i$ are i.i.d. $N(0,1)$,

$$
\frac{\sigma\sqrt{T/N}}{N+1}\sum_{i=1}^{N} iX_i
\sim
N\left(
0,
\frac{(2N+1)\sigma^2T}{6(N+1)}
\right)
$$

By defining the effective volatility as

$$
\tilde{\sigma}
=
\sigma
\sqrt{\frac{2N+1}{6(N+1)}}
$$

we get

$$
\log\left(\frac{\tilde{A}_N}{S(t_0)}\right)
=
(\tilde{r}-\tfrac12\tilde{\sigma}^2)T
+
\tilde{\sigma}\sqrt{T}Z
$$

where

$$
\tilde{r}
=
\left(r-\tfrac12\sigma^2\right)
+
\frac{\tilde{\sigma}^2}{2}
$$

So, the geometric Asian option price becomes ($t_0=0$)

$$
C_g^A(S_0,T)
=
e^{-rT}\mathbb{E}\left[(\tilde{A}_N-K)^+\right]
$$

which can be written as

$$
C_g^A(S_0,T)
=
e^{(\tilde{r}-r)T}C_{BS}(S_0,T;\tilde{r},\tilde{\sigma})
$$

where $C_{BS}$ is the Black–Scholes call price.

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

# Parameters
S0 = 100  # Initial stock price
T = 1     # Time to maturity (1 year)
N = 252    # Number of time intervals for averaging
K = 99    # Strike price
r = 0.06  # Risk-free interest rate
sigma = 0.2 # Volatility
M = 10000   # Number of Monte Carlo simulation paths 

In [ ]:
def geo_asian_analytic(S0, K, T, r, sigma, N):
  
    # adjusted volatility
    sigma_tilde = sigma * np.sqrt((2*N + 1) / (6*(N + 1)))
    
    # adjusted drift
    r_tilde = ((r - 0.5 * sigma**2) + sigma_tilde**2) / 2
    
    # d1 and d2
    d1 = (np.log(S0/K) + (r_tilde + 0.5 * sigma_tilde**2) * T) / (sigma_tilde * np.sqrt(T))
    d2 = (np.log(S0/K) + (r_tilde - 0.5 * sigma_tilde**2) * T) / (sigma_tilde * np.sqrt(T))
    
    # analytical price
    call_price = np.exp(-r*T) * (S0 * np.exp(r_tilde*T) * norm.cdf(d1) - K * norm.cdf(d2))
    
    return call_price

print("Analytical Geometric Asian Call Price:", geo_asian_analytic(S0, K, T, r, sigma, N))

## 2. Monte Carlo Simulation for the Asian Call Option

We simulate stock paths under GBM 
$$
S_{t+\Delta t} = S_t \exp\left(\left(r - \frac{1}{2}\sigma^2 \right)\Delta t + \sigma \sqrt{\Delta_t}Z\right)
$$

In [ ]:
# Monte Carlo geometric Asian option
def mc_asian_geo(S0, K, T, r, sigma, M, N):

    dt = T / N

    Z = np.random.normal(size=(M, N))

    S = np.zeros((M, N))
    S[:,0] = S0 * np.exp((r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z[:,0])

    for t in range(1, N):
        S[:,t] = S[:,t-1] * np.exp((r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z[:,t])

    # geometric average
    geo_avg = np.exp(np.mean(np.log(S), axis=1))

    payoffs = np.maximum(geo_avg - K, 0)

    discounted = np.exp(-r*T) * payoffs

    price = np.mean(discounted)
    std_error = np.std(discounted, ddof=1) / np.sqrt(M)

    return price, std_error, discounted

In [ ]:
# Analytical value
analytical_value = geo_asian_analytic(S0, K, T, r, sigma, N)

# Monte Carlo convergence study
IterationIndex = 100
Iteration = np.linspace(1000, 5000, IterationIndex, dtype=int)

GeoMC = np.zeros((IterationIndex,2))
Analytical_plot = np.zeros(IterationIndex)

for i in range(IterationIndex):

    price, error, _ = mc_asian_geo(S0, K, T, r, sigma, Iteration[i], N)

    GeoMC[i,0] = price
    GeoMC[i,1] = error
    Analytical_plot[i] = analytical_value


plt.figure(figsize=(8, 4))
plt.plot(Iteration, GeoMC[:,0], label="MC Geometric Averages", linewidth=2)
plt.plot(Iteration, Analytical_plot, label="Analytical Geometric Averages", linewidth=2, linestyle='--')
plt.xlabel("Number of paths")
plt.ylabel("Option price")
plt.title("Geometric Asian Option: Monte Carlo vs Analytical")
plt.legend(loc="best")
plt.show()

## 3. Asian Call Option with Arithmetic Averages

This method does not provide an analytical solution like the one from geometric averages. In other words the arithmetic Asian option 
$$
A = \max\left(  \frac{1}{N} \sum_{i=1}^N S_i - K, 0 \right)
$$
has no closed-form solution under Black-Scholes. But, because the arithmetic and geometric averages of the same path are hihgly correlated, we can reduce variance using a **control variate estimator**
$$
\hat{A}_{CV} = A - \beta(G - \mathbb{E}[G])
$$
where 
$$
\beta = \frac{\text{Cov}(A,G)}{\text{Var}(G)}
$$
and $\mathbb{E}[G]$ is the analytical geometric Asian price.

In [ ]:
def mc_asian_arithmetic(S0, K, T, r, sigma, M, N):

    dt = T / N

    Z = np.random.normal(size=(M, N))

    S = np.zeros((M, N))
    S[:,0] = S0 * np.exp((r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z[:,0])

    for t in range(1, N):
        S[:,t] = S[:,t-1] * np.exp((r - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z[:,t])

    # arithmetic average
    arith_avg = np.mean(S, axis=1)

    # payoff
    payoffs = np.maximum(arith_avg - K, 0)

    discounted = np.exp(-r*T) * payoffs

    price = np.mean(discounted)
    std_error = np.std(discounted, ddof=1) / np.sqrt(M)

    return price, std_error, discounted

In [ ]:
# number of experiments
IterationIndex = 80
Iteration = np.linspace(1000,10000,IterationIndex,dtype=int)

GeoMC = np.zeros((IterationIndex,2))
AritMC = np.zeros((IterationIndex,2))
Analytical_plot = np.zeros(IterationIndex)

# analytical geometric value
analytical_value = geo_asian_analytic(S0, K, T, r, sigma, N)

for i in range(IterationIndex):

    M = Iteration[i]

    # geometric MC
    geo_price, geo_error, _ = mc_asian_geo(S0, K, T, r, sigma, M, N)

    # arithmetic MC
    arith_price, arith_error, _ = mc_asian_arithmetic(S0, K, T, r, sigma, M, N)

    GeoMC[i,0] = geo_price
    GeoMC[i,1] = geo_error

    AritMC[i,0] = arith_price
    AritMC[i,1] = arith_error

    Analytical_plot[i] = analytical_value

fig, (plotP, plotQ) = plt.subplots(1,2, figsize=(12,4))

fig.suptitle("Asian Option Pricing: Monte Carlo vs Analytical")

plotP.plot(Iteration, GeoMC[:,0], label="MC Geo Avg")
plotP.plot(Iteration, AritMC[:,0], label="MC Arithmetic Avg")
plotP.plot(Iteration, Analytical_plot, label="Analytical Geometric")
plotP.set_xlabel("Number of paths")
plotP.set_ylabel("Option price")
plotP.legend()

plotQ.plot(Iteration, GeoMC[:,1], label="Geometric Avg MC Std Error")
plotQ.plot(Iteration, AritMC[:,1], label="Arithmetic Avg MC Std Error")
plotQ.set_xlabel("Number of paths")
plotQ.set_ylabel("Standard error")
plotQ.legend()

plt.show()

The $A$ for the arithmetic averages and the $G$ for the geometric averages seem to move together strongly given the above figure. 
$$
\text{Var}(A_{CV}) = \text{Var}(A)(1-\rho^2)
$$
where $\rho$ the correlation between $A$ and $G$.

In [ ]:
# we set the same random seed before each calling to generate the same sequence of random numbers
np.random.seed(42)
geo_price, geo_std, geo_discounted = mc_asian_geo(S0, K, T, r, sigma, M, N)

np.random.seed(42)
arith_price, arith_std, arith_discounted = mc_asian_arithmetic(S0, K, T, r, sigma, M, N)
correlation = np.corrcoef(geo_discounted, arith_discounted)[0, 1]
print(f"Correlation: {correlation:.6f}")

The extremely high correlation of $\rho$ confirms that the arithmetic and geometric averages move almost perfectly together, which means the geometric option as a control variate will result in substantial variance reduction for the arithmetic option price estimator!

In [ ]:
def control_variate(S0, K, T, r, sigma, M, N):

    dt = T/N

    Z = np.random.normal(size=(M,N))

    S = np.zeros((M,N))
    S[:,0] = S0*np.exp((r-0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z[:,0])

    for t in range(1,N):
        S[:,t] = S[:,t-1]*np.exp((r-0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z[:,t])

    # averages
    arith_avg = np.mean(S,axis=1)
    geo_avg = np.exp(np.mean(np.log(S),axis=1))

    # discounted payoffs
    arith_payoff = np.exp(-r*T)*np.maximum(arith_avg-K,0)
    geo_payoff = np.exp(-r*T)*np.maximum(geo_avg-K,0)

    # analytical geometric price
    geo_analytic = geo_asian_analytic(S0,K,T,r,sigma,N)

    # optimal beta
    beta = np.cov(arith_payoff,geo_payoff)[0,1] / np.var(geo_payoff)

    # control variate estimator
    cv_paths = arith_payoff - beta*(geo_payoff - geo_analytic)

    price = np.mean(cv_paths)
    std_error = np.std(cv_paths,ddof=1)/np.sqrt(M)

    return price, std_error

In [ ]:
IterationIndex = 50
Strikegrid = np.linspace(30,200,IterationIndex,dtype=int)

AritMC = np.zeros((IterationIndex,2))
CVMC = np.zeros((IterationIndex,2))

M = 5000

for i in range(IterationIndex):

    K = Strikegrid[i]

    # Arithmetic MC
    arith_price, arith_error, _ = mc_asian_arithmetic(S0, K, T, r, sigma, M, N)

    # Control Variate
    cv_price, cv_error = control_variate(S0, K, T, r, sigma, M, N)

    AritMC[i,0] = arith_price
    AritMC[i,1] = arith_error

    CVMC[i,0] = cv_price
    CVMC[i,1] = cv_error


fig, (plotP, plotQ) = plt.subplots(1, 2, figsize=(14,4))
fig.suptitle('Asian Option Pricing with MC simulation vs Analytical solution')

plotP.plot(Strikegrid, AritMC[:,0], label="Arithmetic MC Avg")
plotP.plot(Strikegrid, CVMC[:,0], label="Control Variate")
plotP.set_xlabel("Strike Price K")
plotP.set_ylabel("Option price")
plotP.legend(loc="best")

plotQ.plot(Strikegrid, CVMC[:,1], label="Control Variate Error", color='orange')
plotQ.set_xlabel("Strike Price K")
plotQ.set_ylabel("Standard error")
plotQ.legend()
plt.show()

In [ ]:
K = 99 
IterationIndex = 80
Paths = np.linspace(1000,20000,IterationIndex,dtype=int)

GeoMC = np.zeros((IterationIndex,2))
AritMC = np.zeros((IterationIndex,2))
CVMC = np.zeros((IterationIndex,2))
Analytical_plot = np.zeros(IterationIndex)

# analytical geometric value
analytical_value = geo_asian_analytic(S0, K, T, r, sigma, N)

for i in range(IterationIndex):

    M = Paths[i]

    # geometric MC
    geo_price, geo_error, _ = mc_asian_geo(S0, K, T, r, sigma, M, N)

    # arithmetic MC
    arith_price, arith_error, _ = mc_asian_arithmetic(S0, K, T, r, sigma, M, N)

    # control variate
    cv_price, cv_error = control_variate(S0, K, T, r, sigma, M, N)

    GeoMC[i,0] = geo_price
    GeoMC[i,1] = geo_error

    AritMC[i,0] = arith_price
    AritMC[i,1] = arith_error

    CVMC[i,0] = cv_price
    CVMC[i,1] = cv_error

    Analytical_plot[i] = analytical_value


fig, (plotP, plotQ) = plt.subplots(1,2, figsize=(14,5))
fig.suptitle("Asian Option Pricing vs Number of Monte Carlo Paths")


plotP.plot(Paths, GeoMC[:,0], label="Geometric MC")
plotP.plot(Paths, AritMC[:,0], label="Arithmetic MC")
plotP.plot(Paths, CVMC[:,0], label="Control Variate")
plotP.plot(Paths, Analytical_plot, label="Analytical Geometric")

plotP.set_xlabel("Number of Paths")
plotP.set_ylabel("Option Price")
plotP.legend()


plotQ.plot(Paths, GeoMC[:,1], label="Geometric MC Error")
plotQ.plot(Paths, AritMC[:,1], label="Arithmetic MC Error")
plotQ.plot(Paths, CVMC[:,1], label="Control Variate Error")

plotQ.set_xlabel("Number of Paths")
plotQ.set_ylabel("Standard Error")
plotQ.legend()

plt.show()